# Sentiment Analysis - Minecraft Reviews (TensorFlow)
**Nama**: Asep Haryana Saputra

## Project Overview
**Objective**: Build a high-accuracy (>92%) sentiment analysis model.
**Dataset**: 40,000 Minecraft reviews.
**Framework**: TensorFlow / Keras.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Bidirectional, GRU, Dropout, GlobalAveragePooling1D
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

print(f"TensorFlow Version: {tf.__version__}")

## 2. Data Loading & Preprocessing
**Labeling Strategy**:
- 1-2 Stars -> **Negative (0)**
- 3 Stars -> **Neutral (1)**
- 4-5 Stars -> **Positive (2)**

In [ ]:
# Load Data
df = pd.read_csv('dataset_raw.csv')
df.dropna(subset=['content'], inplace=True)

# Labeling
def map_sentiment(score):
    if score <= 2: return 0
    elif score == 3: return 1
    else: return 2

df['label'] = df['score'].apply(map_sentiment)

# Clean Text
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['cleaned_content'] = df['content'].apply(clean_text)
print(df['label'].value_counts())

## 3. Tokenization & Splitting

In [ ]:
# Parameters
VOCAB_SIZE = 10000
EMBEDDING_DIM = 100
MAX_LENGTH = 100
TRUNC_TYPE = 'post'
PADDING_TYPE = 'post'
OOV_TOK = "<OOV>"

# Tokenizer
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token=OOV_TOK)
tokenizer.fit_on_texts(df['cleaned_content'])
word_index = tokenizer.word_index

# Sequences
sequences = tokenizer.texts_to_sequences(df['cleaned_content'])
padded = pad_sequences(sequences, maxlen=MAX_LENGTH, padding=PADDING_TYPE, truncating=TRUNC_TYPE)

# Labels to categorical (optional for sparse loss but good for tracking)
labels = np.array(df['label'])

# Split
X_train, X_test, y_train, y_test = train_test_split(padded, labels, test_size=0.2, random_state=42, stratify=labels)
print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")

## 4. Experiment 1: Bi-LSTM Model

In [ ]:
model1 = Sequential([
    Embedding(VOCAB_SIZE, EMBEDDING_DIM, input_length=MAX_LENGTH),
    Bidirectional(LSTM(64, return_sequences=False)),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dense(3, activation='softmax')
])

model1.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
print(model1.summary())

history1 = model1.fit(X_train, y_train, epochs=5, validation_data=(X_test, y_test), batch_size=64)

## 5. Experiment 2: GRU Model

In [ ]:
model2 = Sequential([
    Embedding(VOCAB_SIZE, EMBEDDING_DIM, input_length=MAX_LENGTH),
    GRU(64),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(3, activation='softmax')
])

model2.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
history2 = model2.fit(X_train, y_train, epochs=5, validation_data=(X_test, y_test), batch_size=64)

## 6. Experiment 3: Simple NN (Baseline)

In [ ]:
model3 = Sequential([
    Embedding(VOCAB_SIZE, EMBEDDING_DIM, input_length=MAX_LENGTH),
    GlobalAveragePooling1D(),
    Dense(24, activation='relu'),
    Dense(3, activation='softmax')
])

model3.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
history3 = model3.fit(X_train, y_train, epochs=8, validation_data=(X_test, y_test), batch_size=64)

## 7. Evaluation & Inference

In [ ]:
# Eval Model 1 (Best)
y_pred_prob = model1.predict(X_test)
y_pred = np.argmax(y_pred_prob, axis=1)

print(classification_report(y_test, y_pred, target_names=['Negative', 'Neutral', 'Positive']))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

In [ ]:
# Inference Function
def predict_sentiment(text):
    cleaned = clean_text(text)
    seq = tokenizer.texts_to_sequences([cleaned])
    padded = pad_sequences(seq, maxlen=MAX_LENGTH, padding=PADDING_TYPE, truncating=TRUNC_TYPE)
    pred = model1.predict(padded)
    label_idx = np.argmax(pred)
    
    mapping = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
    return mapping[label_idx]

# Test
print(predict_sentiment("This game is super fun!"))
print(predict_sentiment("Too many bugs, unplayable."))